In [2]:
from QUANTAXIS.QAUtil import DATABASE

In [3]:
from QUANTAXIS.QAFetch.QATdx import (
    QA_fetch_get_option_day,
    QA_fetch_get_option_min,
    QA_fetch_get_index_day,
    QA_fetch_get_index_min,
    QA_fetch_get_stock_day,
    QA_fetch_get_stock_info,
    QA_fetch_get_stock_list,
    QA_fetch_get_future_list,
    QA_fetch_get_index_list,
    QA_fetch_get_future_day,
    QA_fetch_get_future_min,
    QA_fetch_get_stock_min,
    QA_fetch_get_stock_transaction,
    QA_fetch_get_index_transaction,
    QA_fetch_get_stock_xdxr,
    QA_fetch_get_bond_day,
    QA_fetch_get_bond_list,
    QA_fetch_get_bond_min,
    select_best_ip,
    QA_fetch_get_hkstock_day,
    QA_fetch_get_hkstock_list,
    QA_fetch_get_hkstock_min,
    QA_fetch_get_usstock_list,
    QA_fetch_get_usstock_day,
    QA_fetch_get_usstock_min,
)

In [20]:
from QUANTAXIS.QAUtil import (
    DATABASE,
    QA_util_get_next_day,
    QA_util_get_real_date,
    QA_util_log_info,
    QA_util_to_json_from_pandas,
    trade_date_sse
)

In [5]:
from QUANTAXIS.QASU.save_tdx import now_time

In [6]:
import pymongo

In [5]:
def QA_SU_save_stock_day(client=DATABASE, ui_log=None, ui_progress=None):
    '''
     save stock_day
    保存日线数据
    :param client:
    :param ui_log:  给GUI qt 界面使用
    :param ui_progress: 给GUI qt 界面使用
    :param ui_progress_int_value: 给GUI qt 界面使用
    '''
    stock_list = QA_fetch_get_stock_list().code.unique().tolist()
    coll_stock_day = client.stock_day
    coll_stock_day.create_index(
        [("code",
          pymongo.ASCENDING),
         ("date_stamp",
          pymongo.ASCENDING)]
    )
    err = []

    def __saving_work(code, coll_stock_day):
        try:
            QA_util_log_info(
                '##JOB01 Now Saving STOCK_DAY==== {}'.format(str(code)),
                ui_log
            )

            # 首选查找数据库 是否 有 这个代码的数据
            ref = coll_stock_day.find({'code': str(code)[0:6]})
            end_date = str(now_time())[0:10]

            # 当前数据库已经包含了这个代码的数据， 继续增量更新
            # 加入这个判断的原因是因为如果股票是刚上市的 数据库会没有数据 所以会有负索引问题出现
            if ref.count() > 0:

                # 接着上次获取的日期继续更新
                start_date = ref[ref.count() - 1]['date']

                QA_util_log_info(
                    'UPDATE_STOCK_DAY \n Trying updating {} from {} to {}'
                    .format(code,
                            start_date,
                            end_date),
                    ui_log
                )
                if start_date != end_date:
                    coll_stock_day.insert_many(
                        QA_util_to_json_from_pandas(
                            QA_fetch_get_stock_day(
                                str(code),
                                QA_util_get_next_day(start_date),
                                end_date,
                                '00'
                            )
                        )
                    )

            # 当前数据库中没有这个代码的股票数据， 从1990-01-01 开始下载所有的数据
            else:
                start_date = '1990-01-01'
                QA_util_log_info(
                    'UPDATE_STOCK_DAY \n Trying updating {} from {} to {}'
                    .format(code,
                            start_date,
                            end_date),
                    ui_log
                )
                if start_date != end_date:
                    coll_stock_day.insert_many(
                        QA_util_to_json_from_pandas(
                            QA_fetch_get_stock_day(
                                str(code),
                                start_date,
                                end_date,
                                '00'
                            )
                        )
                    )
        except Exception as error0:
            print(error0)
            err.append(str(code))

    for item in range(len(stock_list)):
        QA_util_log_info('The {} of Total {}'.format(item, len(stock_list)))

        strProgressToLog = 'DOWNLOAD PROGRESS {} {}'.format(
            str(float(item / len(stock_list) * 100))[0:4] + '%',
            ui_log
        )
        intProgressToLog = int(float(item / len(stock_list) * 100))
        QA_util_log_info(
            strProgressToLog,
            ui_log=ui_log,
            ui_progress=ui_progress,
            ui_progress_int_value=intProgressToLog
        )

        __saving_work(stock_list[item], coll_stock_day)

    if len(err) < 1:
        QA_util_log_info('SUCCESS save stock day ^_^', ui_log)
    else:
        QA_util_log_info('ERROR CODE \n ', ui_log)
        QA_util_log_info(err, ui_log)

In [ ]:
QA_SU_save_stock_day(client=DATABASE,ui_log=None, ui_progress=None)

QUANTAXIS>> The 0 of Total 5205
QUANTAXIS>> DOWNLOAD PROGRESS 0.0% None
QUANTAXIS>> ##JOB01 Now Saving STOCK_DAY==== 000001
<ipython-input-5-431088e8f570>:33: DeprecationWarning: count is deprecated. Use Collection.count_documents instead.
  if ref.count() > 0:
<ipython-input-5-431088e8f570>:36: DeprecationWarning: count is deprecated. Use Collection.count_documents instead.
  start_date = ref[ref.count() - 1]['date']
QUANTAXIS>> UPDATE_STOCK_DAY 
 Trying updating 000001 from 2025-05-23 to 2026-07-10
QUANTAXIS>> The 1 of Total 5205
QUANTAXIS>> DOWNLOAD PROGRESS 0.01% None
QUANTAXIS>> ##JOB01 Now Saving STOCK_DAY==== 000002
<ipython-input-5-431088e8f570>:33: DeprecationWarning: count is deprecated. Use Collection.count_documents instead.
  if ref.count() > 0:
<ipython-input-5-431088e8f570>:36: DeprecationWarning: count is deprecated. Use Collection.count_documents instead.
  start_date = ref[ref.count() - 1]['date']
QUANTAXIS>> UPDATE_STOCK_DAY 
 Trying updating 000002 from 2025-05-23 t

name 'date' is not defined


QUANTAXIS>> The 2 of Total 5205
QUANTAXIS>> DOWNLOAD PROGRESS 0.03% None
QUANTAXIS>> ##JOB01 Now Saving STOCK_DAY==== 000004
<ipython-input-5-431088e8f570>:33: DeprecationWarning: count is deprecated. Use Collection.count_documents instead.
  if ref.count() > 0:
<ipython-input-5-431088e8f570>:36: DeprecationWarning: count is deprecated. Use Collection.count_documents instead.
  start_date = ref[ref.count() - 1]['date']
QUANTAXIS>> UPDATE_STOCK_DAY 
 Trying updating 000004 from 2025-05-23 to 2026-07-10


name 'date' is not defined


QUANTAXIS>> The 3 of Total 5205
QUANTAXIS>> DOWNLOAD PROGRESS 0.05% None
QUANTAXIS>> ##JOB01 Now Saving STOCK_DAY==== 000006
<ipython-input-5-431088e8f570>:33: DeprecationWarning: count is deprecated. Use Collection.count_documents instead.
  if ref.count() > 0:
<ipython-input-5-431088e8f570>:36: DeprecationWarning: count is deprecated. Use Collection.count_documents instead.
  start_date = ref[ref.count() - 1]['date']
QUANTAXIS>> UPDATE_STOCK_DAY 
 Trying updating 000006 from 2025-05-23 to 2026-07-10


...

QUANTAXIS>> The 2300 of Total 5205
QUANTAXIS>> DOWNLOAD PROGRESS 44.1% None
QUANTAXIS>> ##JOB01 Now Saving STOCK_DAY==== 300861
<ipython-input-5-431088e8f570>:33: DeprecationWarning: count is deprecated. Use Collection.count_documents instead.
  if ref.count() > 0:
<ipython-input-5-431088e8f570>:36: DeprecationWarning: count is deprecated. Use Collection.count_documents instead.
  start_date = ref[ref.count() - 1]['date']
QUANTAXIS>> UPDATE_STOCK_DAY 
 Trying updating 300861 from 2025-05-16 to 2026-07-10


name 'date' is not defined


QUANTAXIS>> The 2301 of Total 5205
QUANTAXIS>> DOWNLOAD PROGRESS 44.2% None
QUANTAXIS>> ##JOB01 Now Saving STOCK_DAY==== 300862
<ipython-input-5-431088e8f570>:33: DeprecationWarning: count is deprecated. Use Collection.count_documents instead.
  if ref.count() > 0:
<ipython-input-5-431088e8f570>:36: DeprecationWarning: count is deprecated. Use Collection.count_documents instead.
  start_date = ref[ref.count() - 1]['date']
QUANTAXIS>> UPDATE_STOCK_DAY 
 Trying updating 300862 from 2025-05-23 to 2026-07-10


name 'date' is not defined


QUANTAXIS>> The 2302 of Total 5205
QUANTAXIS>> DOWNLOAD PROGRESS 44.2% None
QUANTAXIS>> ##JOB01 Now Saving STOCK_DAY==== 300863
<ipython-input-5-431088e8f570>:33: DeprecationWarning: count is deprecated. Use Collection.count_documents instead.
  if ref.count() > 0:
<ipython-input-5-431088e8f570>:36: DeprecationWarning: count is deprecated. Use Collection.count_documents instead.
  start_date = ref[ref.count() - 1]['date']
QUANTAXIS>> UPDATE_STOCK_DAY 
 Trying updating 300863 from 2025-05-23 to 2026-07-10


In [7]:
QA_fetch_get_stock_list().code.unique().tolist()

QUANTAXIS>> Selecting the Best Server IP of TDX
QUANTAXIS>> === The BEST SERVER ===
 stock_ip shtdx.gtjas.com future_ip 112.74.214.43


USING DEFAULT STOCK IP
USING DEFAULT FUTURE IP


['000001',
 '000002',
 '000004',
 '000006',
 '000007',
 '000008',
 '000009',
 '000010',
 '000011',
 '000012',
 '000014',
 '000016',
 '000017',
 '000019',
 '000020',
 '000021',
 '000025',
 '000026',
 '000027',
 '000028',
 '000029',
 '000030',
 '000031',
 '000032',
 '000034',
 '000035',
 '000036',
 '000037',
 '000039',
 '000042',
 '000045',
 '000048',
 '000049',
 '000050',
 '000055',
 '000056',
 '000058',
 '000059',
 '000060',
 '000061',
 '000062',
 '000063',
 '000065',
 '000066',
 '000068',
 '000069',
 '000070',
 '000078',
 '000088',
 '000089',
 '000090',
 '000096',
 '000099',
 '000100',
 '000151',
 '000153',
 '000155',
 '000156',
 '000157',
 '000158',
 '000159',
 '000166',
 '000301',
 '000333',
 '000338',
 '000400',
 '000401',
 '000402',
 '000403',
 '000404',
 '000407',
 '000408',
 '000409',
 '000410',
 '000411',
 '000415',
 '000417',
 '000419',
 '000420',
 '000421',
 '000422',
 '000423',
 '000425',
 '000426',
 '000428',
 '000429',
 '000430',
 '000488',
 '000498',
 '000501',
 '000503',

In [8]:
# stock_list = QA_fetch_get_stock_list().code.unique().tolist()
client = DATABASE
stock_list =  ['000001', '000002']
coll_stock_day = client.stock_day
coll_stock_day.create_index(
    [("code",
      pymongo.ASCENDING),
     ("date_stamp",
      pymongo.ASCENDING)]
)
err = []

# def __saving_work(code, coll_stock_day):
#     try:
#         QA_util_log_info(
#             '##JOB01 Now Saving STOCK_DAY==== {}'.format(str(code)),
#             ui_log
#         )

#         # 首选查找数据库 是否 有 这个代码的数据
#         ref = coll_stock_day.find({'code': str(code)[0:6]})
#         end_date = str(now_time())[0:10]

#         # 当前数据库已经包含了这个代码的数据， 继续增量更新
#         # 加入这个判断的原因是因为如果股票是刚上市的 数据库会没有数据 所以会有负索引问题出现
#         if ref.count() > 0:

#             # 接着上次获取的日期继续更新
#             start_date = ref[ref.count() - 1]['date']

#             QA_util_log_info(
#                 'UPDATE_STOCK_DAY \n Trying updating {} from {} to {}'
#                 .format(code,
#                         start_date,
#                         end_date),
#                 ui_log
#             )
#             if start_date != end_date:
#                 coll_stock_day.insert_many(
#                     QA_util_to_json_from_pandas(
#                         QA_fetch_get_stock_day(
#                             str(code),
#                             QA_util_get_next_day(start_date),
#                             end_date,
#                             '00'
#                         )
#                     )
#                 )

#         # 当前数据库中没有这个代码的股票数据， 从1990-01-01 开始下载所有的数据
#         else:
#             start_date = '1990-01-01'
#             QA_util_log_info(
#                 'UPDATE_STOCK_DAY \n Trying updating {} from {} to {}'
#                 .format(code,
#                         start_date,
#                         end_date),
#                 ui_log
#             )
#             if start_date != end_date:
#                 coll_stock_day.insert_many(
#                     QA_util_to_json_from_pandas(
#                         QA_fetch_get_stock_day(
#                             str(code),
#                             start_date,
#                             end_date,
#                             '00'
#                         )
#                     )
#                 )
#     except Exception as error0:
#         print(error0)
#         err.append(str(code))

# for item in range(len(stock_list)):
#     QA_util_log_info('The {} of Total {}'.format(item, len(stock_list)))

#     strProgressToLog = 'DOWNLOAD PROGRESS {} {}'.format(
#         str(float(item / len(stock_list) * 100))[0:4] + '%',
#         ui_log
#     )
#     intProgressToLog = int(float(item / len(stock_list) * 100))
#     QA_util_log_info(
#         strProgressToLog,
#         ui_log=ui_log,
#         ui_progress=ui_progress,
#         ui_progress_int_value=intProgressToLog
#     )

#     __saving_work(stock_list[item], coll_stock_day)

# if len(err) < 1:
#     QA_util_log_info('SUCCESS save stock day ^_^', ui_log)
# else:
#     QA_util_log_info('ERROR CODE \n ', ui_log)
#     QA_util_log_info(err, ui_log)

'code_1_date_stamp_1'

In [24]:
import json

code = '000001'
# 首选查找数据库 是否 有 这个代码的数据
ref = coll_stock_day.find({'code': str(code)[0:6]})
end_date = str(now_time())[0:10]
ui_log = None

# 当前数据库已经包含了这个代码的数据， 继续增量更新
# 加入这个判断的原因是因为如果股票是刚上市的 数据库会没有数据 所以会有负索引问题出现
# if ref.count() > 0:

# 接着上次获取的日期继续更新
start_date = ref[ref.count() - 1]['date']

QA_util_log_info(
    'UPDATE_STOCK_DAY \n Trying updating {} from {} to {}'
    .format(code,
            start_date,
            end_date),
    ui_log
)
if start_date != end_date:
    coll_stock_day.insert_many(
        QA_util_to_json_from_pandas(
            QA_fetch_get_stock_day(
                str(code),
                QA_util_get_next_day(start_date),
                end_date,
                '00'
            )
        )
    )

# # 当前数据库中没有这个代码的股票数据， 从1990-01-01 开始下载所有的数据
# else:
#     start_date = '1990-01-01'
#     QA_util_log_info(
#         'UPDATE_STOCK_DAY \n Trying updating {} from {} to {}'
#         .format(code,
#                 start_date,
#                 end_date),
#         ui_log
#     )
#     if start_date != end_date:
#         coll_stock_day.insert_many(
#             QA_util_to_json_from_pandas(
#                 QA_fetch_get_stock_day(
#                     str(code),
#                     start_date,
#                     end_date,
#                     '00'
#                 )
#             )
#         )

<ipython-input-24-2a2e3eb7c02c>:14: DeprecationWarning: count is deprecated. Use Collection.count_documents instead.
  start_date = ref[ref.count() - 1]['date']
QUANTAXIS>> UPDATE_STOCK_DAY 
 Trying updating 000001 from 2025-05-23 to 2026-07-10


In [22]:
def QA_util_to_json_from_pandas(data):
    """
    explanation:
        将pandas数据转换成json格式		

    params:
        * data ->:
            meaning: pandas数据
            type: null
            optional: [null]

    return:
        dict

    demonstrate:
        Not described

    output:
        Not described
    """

    """需要对于datetime 和date 进行转换, 以免直接被变成了时间戳"""

    #  .loc[row_indexer,col_indexer]=
    if 'datetime' in data.columns:
        # data.datetime = data.datetime.apply(str)
        data.loc[:, 'datetime'] = data.loc[:, 'datetime'].apply(str)
    if 'date' in data.columns:
        # data.date = data.date.apply(str)
        data.loc[:, 'date'] = data.loc[:, 'date'].apply(str)
    return json.loads(data.to_json(orient='records'))

In [14]:
coll_stock_day.count_documents({'code': str(code)[0:6]})

8132

In [26]:
coll_stock_day.count_documents({'code': str(code)[0:6]})

8407